In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import os
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb
!pip install catboost
from catboost import CatBoostClassifier
import plotly.graph_objects as go


warnings.filterwarnings("ignore")

In [14]:
DATASETS = ['TICK1.csv',
            'TICK2.csv',
            'TICK3.csv',
            'TICK4.csv']

TARGET_VOL = 0.25
BOOST_FACTOR = 3
BOOST_THRESHOLD = 0.20
MIN_SIGNAL = 0.10
STOP_ATR_K = 2
TIME_EXIT = 10
FEE_BPS = 3.0
WARMUP = 200
LOOKBACK_VOL = 40

In [15]:
def load_and_normalize(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()  # remove extra spaces
    df["date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["open"] = df["Open"].astype(float)
    df["high"] = df["High"].astype(float)
    df["low"] = df["Low"].astype(float)
    df["close"] = df["Close"].astype(float)
    df["volume"] = df["Volume"].astype(float)

    df = df.sort_values("date").reset_index(drop=True)
    df["ret"] = df["close"].pct_change()
    df = df.dropna().reset_index(drop=True)

    return df[["date", "open", "high", "low", "close", "volume", "ret"]]

In [16]:
def train_and_evaluate_with_cv(X, y, n_splits=5):
    cv = TimeSeriesSplit(n_splits=n_splits)

    # Initialize dictionaries to track performance for each model
    accuracy_list = {"logreg": [], "xgb": [], "lgb": []}
    auc_list = {"logreg": [], "xgb": [], "lgb": []}

    # Define models
    models = {
        "logreg": LogisticRegression(max_iter=500, penalty="l2", solver='lbfgs', random_state=42),  # Logistic Regression
        "xgb": xgb.XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42),  # XGBoost
        "lgb": lgb.LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42, verbosity=-1),  # LightGBM
    }

    # Perform Stratified K-Fold cross-validation
    for train_idx, test_idx in cv.split(X, y):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Train and evaluate each model
        for model_name, model in models.items():
            model.fit(X_train, y_train)  # Train the model
            y_pred = model.predict(X_test)  # Predict the labels
            y_pred_prob = model.predict_proba(X_test)[:, 1]  # Probabilities for AUC calculation

            accuracy = accuracy_score(y_test, y_pred)  # Calculate accuracy
            auc = roc_auc_score(y_test, y_pred_prob)  # Calculate AUC

            # Append the results to the corresponding model's list
            accuracy_list[model_name].append(accuracy)
            auc_list[model_name].append(auc)

    # Calculate mean and std for accuracy and AUC for each model
    metrics = {}
    for model_name in models.keys():
        metrics[model_name] = {
            "mean_accuracy": np.mean(accuracy_list[model_name]),
            "std_accuracy": np.std(accuracy_list[model_name]),
            "mean_auc": np.mean(auc_list[model_name]),
            "std_auc": np.std(auc_list[model_name])
        }

    return metrics

In [17]:
def train_models(X_train, y_train):
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_train)

    models = {
        "logreg": LogisticRegression(max_iter=500, random_state=42),
        "xgb": xgb.XGBClassifier(n_estimators=100, max_depth=3, random_state=42),
        "lgb": lgb.LGBMClassifier(n_estimators=100, max_depth=3, random_state=42),
    }

    for m in models.values():
        m.fit(X_scaled, y_train)

    return models, scaler

def ensemble_probs(models, scaler, X):
    X_scaled = scaler.transform(X)
    probs = np.zeros(len(X))
    for m in models.values():
        probs += m.predict_proba(X_scaled)[:, 1]
    return probs / len(models)

In [18]:
def atr(df, period=14):
    df = df.copy()
    df['prev_close'] = df['close'].shift(1)
    tr1 = df['high'] - df['low']
    tr2 = (df['high'] - df['prev_close']).abs()
    tr3 = (df['low'] - df['prev_close']).abs()
    df['TR'] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    df['ATR'] = df['TR'].ewm(span=period, adjust=False).mean()
    return df['ATR']  # Return just the ATR series

def compute_keltner_bands(df, period=20, multiplier=1):
    df = df.copy()

    # Compute KC_EMA using the closing price
    df['KC_EMA'] = df['close'].ewm(span=period, adjust=False).mean()

    # Calculate ATR and return it to the DataFrame
    df['ATR'] = atr(df, period=period)  # Now 'ATR' is directly added to df

    # Calculate the Keltner Bands
    df['KC_Upper'] = df['KC_EMA'] + multiplier * df['ATR']
    df['KC_Lower'] = df['KC_EMA'] - multiplier * df['ATR']

    # Signal generation for Keltner Bands
    signal = [0] * len(df)
    last_sig = 0
    for i in range(len(df)):
        if df['close'].iloc[i] > df['KC_Upper'].iloc[i]:
            sig = 1
        elif df['close'].iloc[i] < df['KC_Lower'].iloc[i]:
            sig = -1
        else:
            sig = 0
        if sig == last_sig:
            sig = 0
        else:
            if sig != 0:
                last_sig = sig
        signal[i] = sig
    df['KC_signal'] = signal

    return df

def add_indicators(df, atr_window=15, multiplier=1):
    df["sma_50"] = df["close"].rolling(window=50).mean()
    df["sma_200"] = df["close"].rolling(window=200).mean()
    df["ema_50"] = df["close"].ewm(span=50, adjust=False).mean()
    df["ema_200"] = df["close"].ewm(span=200, adjust=False).mean()

    # Compute and add Keltner Bands and other indicators
    df = compute_keltner_bands(df, period=atr_window, multiplier=multiplier)

    return df

In [19]:
def boost_prob(p, factor=BOOST_FACTOR):
    return np.tanh((p - 0.5) * factor)

# Compute Volatility Scale
def compute_vol_scale(raw_signal, asset_rets, lookback=LOOKBACK_VOL, target_vol=TARGET_VOL, min_floor=1e-3, smooth_window=10):
    raw = pd.Series(raw_signal, index=asset_rets.index).fillna(0.0)
    strat_rets = raw * asset_rets
    roll_vol = strat_rets.rolling(lookback).std() * np.sqrt(252)
    roll_vol = roll_vol.fillna(method="bfill").clip(lower=min_floor)
    raw_scale = target_vol / roll_vol
    smoothed_scale = raw_scale.rolling(smooth_window).mean().fillna(0.0)
    return smoothed_scale.shift(1).fillna(0.0)

In [20]:
def backtest(df, targets, fee_bps=FEE_BPS, stop_loss_pct=0.05,
             vol_lookback=20, scale_thresh=0.30, scale_factor=0.75, stop_atr_k=1.5):
    df = df.reset_index(drop=True)
    dates = df["date"].values
    opens = df["open"].values
    closes = df["close"].values
    atrs = atr(df, 14).fillna(method="bfill").values

    # Rolling mean/std returns for capital scaling
    returns = pd.Series(df["close"]).pct_change().fillna(0.0)
    rolling_mean_ret = returns.rolling(vol_lookback).mean().fillna(0.0)
    rolling_std_ret = rolling_mean_ret.rolling(vol_lookback).std().fillna(0.0)

    equity = 1_000_000.0
    prev_target = 0.0
    entry_price = None
    entry_idx = None
    pos = 0.0
    stop_loss_price = None
    rows = []

    for i in range(1, len(df)):
        raw_target = float(targets.iloc[i-1])

        # Capital scaling based on rolling std of mean returns
        capital_scale = scale_factor if rolling_std_ret[i-1] > scale_thresh else 1.0
        capital_for_trade = equity * capital_scale

        target = raw_target  # signal direction (-1 to 1)

        # Compute ATR-based stop distance
        stop_dist = stop_atr_k * atrs[i-1] if atrs[i-1] > 0 else 0.0

        # Trailing Stop-Loss based on ATR and fixed stop-loss
        if entry_price is not None:
            if stop_loss_pct is not None:
                stop_loss_price = entry_price * (1 - stop_loss_pct) if pos > 0 else entry_price * (1 + stop_loss_pct)

            # Trailing Stop-Loss logic
            if pos > 0 and (closes[i-1] <= entry_price - stop_dist or closes[i-1] <= stop_loss_price):
                target = 0.0
                entry_price = None
                entry_idx = None
            elif pos < 0 and (closes[i-1] >= entry_price + stop_dist or closes[i-1] >= stop_loss_price):
                target = 0.0
                entry_price = None
                entry_idx = None

        # Fees and P&L
        turnover = abs(target - prev_target)
        fees = turnover * equity * (fee_bps / 1e4)

        open_px = opens[i]
        close_px = closes[i]

        # Position units based on scaled capital
        pos_units = (target * capital_for_trade) / open_px if open_px > 0 else 0.0
        pnl = pos_units * (close_px - open_px)
        equity = equity + pnl - fees

        # Track entry/exit for stop-loss
        if entry_price is None and abs(target) > 1e-9:
            entry_price = open_px
            entry_idx = i
            stop_loss_price = None
        if abs(target) < 1e-9:
            entry_price = None
            entry_idx = None
            stop_loss_price = None

        prev_target = target
        pos = target
        rows.append([dates[i], equity])

    return pd.DataFrame(rows, columns=["date", "equity"]).set_index("date")


In [21]:
def compute_metrics(df, risk_free_rate=0.06/365):
    df = df.dropna()

    if len(df) < 3:
        return {
            "Annual Return": np.nan, "Annual Volatility": np.nan, "Sharpe": np.nan, "Sortino": np.nan,
            "Max Drawdown": np.nan, "CAGR": np.nan, "Win Rate": np.nan, "Total Trades": np.nan,
            "Winning Trades": np.nan, "Losing Trades": np.nan, "Avg Win %": np.nan,
            "Avg Holding Days": np.nan, "Calmar Ratio": np.nan, "Gross Profit": np.nan, "Net Profit": np.nan
        }

    rets = df["equity"].pct_change().dropna()
    ann_ret = (df["equity"].iloc[-1] / df["equity"].iloc[0]) ** (252 / len(rets)) - 1
    ann_vol = rets.std() * np.sqrt(252)

    # Sharpe Ratio
    sharpe = (rets.mean() - risk_free_rate) / (rets.std() + 1e-12) * np.sqrt(252)

    # Downside Deviation
    downside = rets[rets < 0].std() * np.sqrt(252)
    sortino = ann_ret / downside if downside > 0 else np.nan

    # Maximum Drawdown
    cummax = df["equity"].cummax()
    mdd = ((df["equity"] - cummax) / cummax).min()

    # Compound Annual Growth Rate (CAGR)
    cagr = (df["equity"].iloc[-1] / df["equity"].iloc[0]) ** (1 / (len(df) / 252)) - 1

    # Calmar Ratio (Annual Return / Maximum Drawdown)
    calmar_ratio = ann_ret / abs(mdd) if mdd != 0 else np.nan

    # Trade statistics
    trades = df["equity"].pct_change().dropna()
    gross_profit = trades[trades > 0].sum() * 100  # as a percentage
    gross_loss = trades[trades < 0].sum() * 100  # as a percentage
    net_profit = gross_profit + gross_loss
    break_even_trades = len(trades[trades == 0])
    total_trades = len(trades[trades != 0])
    winning_trades = len(trades[trades > 0])
    losing_trades = len(trades[trades < 0])
    win_rate = winning_trades / total_trades * 100 if total_trades > 0 else 0

    # Avg Win % and Avg Holding Days
    avg_win = trades[trades > 0].mean() * 100 if winning_trades > 0 else 0
    avg_loss = trades[trades < 0].mean() * 100 if losing_trades > 0 else 0
    avg_holding = len(df) / total_trades if total_trades > 0 else 0

    # Print the results
    print(f"Gross Profit: {gross_profit:.2f} %")
    print(f"Total Trades: {total_trades}")
    print(f"Winning Trades: {winning_trades}")
    print(f"Losing Trades: {losing_trades}")
    print(f"Win Rate: {win_rate:.2f} %")
    print(f"Max Drawdown: {mdd:.2f} %")
    print(f"Sharpe Ratio: {sharpe:.2f}")
    print(f"CAGR: {cagr:.2f} %")
    print(f"Sortino Ratio: {sortino:.2f}")
    print(f"Calmar Ratio: {calmar_ratio:.2f}")
    print(f"Avg Win %: {avg_win:.2f}")
    print(f"Avg Holding Days: {avg_holding:.2f}")

    return {
        "Annual Return": ann_ret,
        "Annual Volatility": ann_vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Max Drawdown": mdd,
        "CAGR": cagr,
        "Win Rate": win_rate,
        "Total Trades": total_trades,
        "Winning Trades": winning_trades,
        "Losing Trades": losing_trades,
        "Avg Win %": avg_win,
        "Avg Holding Days": avg_holding,
        "Calmar Ratio": calmar_ratio,
        "Gross Profit": gross_profit,
        "Net Profit": net_profit
    }

In [22]:
import plotly.graph_objects as go
summary = {}

for fname in DATASETS:
    print(f"\n--- Processing {fname} ---")
    df_raw = load_and_normalize(fname)

    # Generate features based on ATR and Keltner Channel
    df = add_indicators(df_raw)

    # Define the columns for the model input (features)
    feat_cols = [
    "ATR_signal", "KC_signal", "sma_50", "sma_200", "ema_50", "ema_200",  # Indicators
    "KC_Upper", "KC_Lower",  # Keltner Bands
    "ATR_Upper", "ATR_Lower"  # ATR Bands
]

    # Ensure all the required columns are present
    for c in feat_cols:
        if c not in df.columns:
            df[c] = 0.0

    # Prepare features and labels
    X = df[feat_cols].fillna(0.0)
    y = (df["ret"].shift(-1) > 0).astype(int).fillna(0).astype(int)

    # Require minimal data
    if len(df) < WARMUP:
        print(f"{fname}: not enough data (need >= {WARMUP} rows).")
        summary[fname] = {"error": "not enough data"}
        continue

    # Split into 80% train / 20% test
    n = len(df)
    train_idx = int(0.8 * n)
    X_train, X_test = X.iloc[:train_idx], X.iloc[train_idx:]
    y_train, y_test = y.iloc[:train_idx], y.iloc[train_idx:]
    df_test = df.iloc[train_idx:]

    metrics = train_and_evaluate_with_cv(X, y, n_splits=5)
    print("CV Performance Summary:\n")
    for model_name, vals in metrics.items():
        print(f"{model_name.capitalize()}:")
        print(f"  Mean Accuracy : {vals['mean_accuracy']:.4f}")
        print(f"  Std Accuracy  : {vals['std_accuracy']:.4f}")
        print(f"  Mean AUC      : {vals['mean_auc']:.4f}")
        print(f"  Std AUC       : {vals['std_auc']:.4f}\n")

    # Train the models on the first (len-1) rows and predict on the full series (simple approach)
    models, scaler = train_models(X.iloc[:len(X)-1], y.iloc[:len(X)-1])

    # Generate ensemble probabilities across all rows (we use all but the last for signals)
    probs = ensemble_probs(models, scaler, X)

    # Boost probabilities with a factor
    boosted = boost_prob(probs, factor=BOOST_FACTOR)
    boosted_series = pd.Series(boosted, index=df.index)

    # Threshold small signals
    boosted_series[np.abs(boosted_series) < BOOST_THRESHOLD] = 0.0

    # Compute volatility scaling (this aligns with the DataFrame)
    scale = compute_vol_scale(boosted_series, df["ret"], lookback=LOOKBACK_VOL, target_vol=TARGET_VOL)
    targets = boosted_series * scale
    targets = targets.fillna(0.0)

    # Perform backtesting
    # eq = backtest(df, targets, fee_bps=FEE_BPS, stop_atr_k=STOP_ATR_K)
    eq = backtest(
    df,
    targets,
    fee_bps=FEE_BPS,
    stop_loss_pct=0.05,
    vol_lookback=20,
    scale_thresh=0.0005,
    scale_factor=0.75
)
    # Calculate performance metrics
    mets = compute_metrics(eq)
    summary[fname] = {"metrics": mets, "equity": eq, "targets": targets, "scale": scale}

    # Print metrics
    print("Metrics:")
    for k, v in mets.items():
        print(f"  {k}: {v:.4f}" if (v == v) else f"  {k}: {v}")

    # Plot equity curve
    fig = go.Figure()

    fig.add_trace(go.Scatter(x=eq.index, y=eq["equity"], mode="lines", name=f"Equity - {fname}"))
    fig.update_layout(
        title=f"Equity Curve - {fname}",
        xaxis_title="Date",
        yaxis_title="Equity",
        template="plotly_dark"
    )
    fig.show()


--- Processing TICK1.csv ---
CV Performance Summary:

Logreg:
  Mean Accuracy : 0.5148
  Std Accuracy  : 0.0432
  Mean AUC      : 0.5170
  Std AUC       : 0.0616

Xgb:
  Mean Accuracy : 0.5025
  Std Accuracy  : 0.0296
  Mean AUC      : 0.5039
  Std AUC       : 0.0435

Lgb:
  Mean Accuracy : 0.4963
  Std Accuracy  : 0.0403
  Mean AUC      : 0.5059
  Std AUC       : 0.0446

Gross Profit: 502.12 %
Total Trades: 644
Winning Trades: 376
Losing Trades: 268
Win Rate: 58.39 %
Max Drawdown: -0.10 %
Sharpe Ratio: 4.80
CAGR: 1.63 %
Sortino Ratio: 14.99
Calmar Ratio: 17.16
Avg Win %: 1.34
Avg Holding Days: 1.51
Metrics:
  Annual Return: 1.6314
  Annual Volatility: 0.1972
  Sharpe: 4.8022
  Sortino: 14.9890
  Max Drawdown: -0.0950
  CAGR: 1.6288
  Win Rate: 58.3851
  Total Trades: 644.0000
  Winning Trades: 376.0000
  Losing Trades: 268.0000
  Avg Win %: 1.3354
  Avg Holding Days: 1.5140
  Calmar Ratio: 17.1638
  Gross Profit: 502.1211
  Net Profit: 381.9894



--- Processing TICK2.csv ---
CV Performance Summary:

Logreg:
  Mean Accuracy : 0.5284
  Std Accuracy  : 0.0634
  Mean AUC      : 0.4811
  Std AUC       : 0.0512

Xgb:
  Mean Accuracy : 0.5284
  Std Accuracy  : 0.0340
  Mean AUC      : 0.5093
  Std AUC       : 0.0512

Lgb:
  Mean Accuracy : 0.5297
  Std Accuracy  : 0.0359
  Mean AUC      : 0.4861
  Std AUC       : 0.0572

Gross Profit: 1054.05 %
Total Trades: 591
Winning Trades: 377
Losing Trades: 214
Win Rate: 63.79 %
Max Drawdown: -0.28 %
Sharpe Ratio: 1.05
CAGR: 4.40 %
Sortino Ratio: 10.72
Calmar Ratio: 15.80
Avg Win %: 2.80
Avg Holding Days: 1.50
Metrics:
  Annual Return: 4.4080
  Annual Volatility: 2.3896
  Sharpe: 1.0549
  Sortino: 10.7226
  Max Drawdown: -0.2789
  CAGR: 4.3978
  Win Rate: 63.7902
  Total Trades: 591.0000
  Winning Trades: 377.0000
  Losing Trades: 214.0000
  Avg Win %: 2.7959
  Avg Holding Days: 1.5042
  Calmar Ratio: 15.8030
  Gross Profit: 1054.0550
  Net Profit: 902.8892



--- Processing TICK3.csv ---
CV Performance Summary:

Logreg:
  Mean Accuracy : 0.5338
  Std Accuracy  : 0.0310
  Mean AUC      : 0.5199
  Std AUC       : 0.0092

Xgb:
  Mean Accuracy : 0.4924
  Std Accuracy  : 0.0221
  Mean AUC      : 0.4993
  Std AUC       : 0.0120

Lgb:
  Mean Accuracy : 0.5076
  Std Accuracy  : 0.0348
  Mean AUC      : 0.4933
  Std AUC       : 0.0075

Gross Profit: 515.50 %
Total Trades: 619
Winning Trades: 398
Losing Trades: 221
Win Rate: 64.30 %
Max Drawdown: -0.05 %
Sharpe Ratio: 5.90
CAGR: 2.30 %
Sortino Ratio: 24.58
Calmar Ratio: 46.35
Avg Win %: 1.30
Avg Holding Days: 1.41
Metrics:
  Annual Return: 2.3021
  Annual Volatility: 0.1992
  Sharpe: 5.9000
  Sortino: 24.5829
  Max Drawdown: -0.0497
  CAGR: 2.2975
  Win Rate: 64.2973
  Total Trades: 619.0000
  Winning Trades: 398.0000
  Losing Trades: 221.0000
  Avg Win %: 1.2952
  Avg Holding Days: 1.4071
  Calmar Ratio: 46.3465
  Gross Profit: 515.5013
  Net Profit: 420.0409



--- Processing TICK4.csv ---
CV Performance Summary:

Logreg:
  Mean Accuracy : 0.5323
  Std Accuracy  : 0.0554
  Mean AUC      : 0.5352
  Std AUC       : 0.0584

Xgb:
  Mean Accuracy : 0.5323
  Std Accuracy  : 0.0502
  Mean AUC      : 0.5496
  Std AUC       : 0.0622

Lgb:
  Mean Accuracy : 0.5371
  Std Accuracy  : 0.0359
  Mean AUC      : 0.5349
  Std AUC       : 0.0446

Gross Profit: 343.61 %
Total Trades: 479
Winning Trades: 230
Losing Trades: 249
Win Rate: 48.02 %
Max Drawdown: -0.39 %
Sharpe Ratio: 1.95
CAGR: 0.77 %
Sortino Ratio: 2.51
Calmar Ratio: 1.98
Avg Win %: 1.49
Avg Holding Days: 1.55
Metrics:
  Annual Return: 0.7712
  Annual Volatility: 0.2940
  Sharpe: 1.9543
  Sortino: 2.5086
  Max Drawdown: -0.3896
  CAGR: 0.7698
  Win Rate: 48.0167
  Total Trades: 479.0000
  Winning Trades: 230.0000
  Losing Trades: 249.0000
  Avg Win %: 1.4940
  Avg Holding Days: 1.5532
  Calmar Ratio: 1.9794
  Gross Profit: 343.6092
  Net Profit: 181.6252


In [23]:
print("\n=== Summary Table ===")
rows = []
for k, v in summary.items():
    if "metrics" in v:
        m = v["metrics"]
        # Append the additional metrics to the row
        rows.append([
            k,
            m.get("Annual Return", None),
            m.get("Annual Volatility", None),
            m.get("Sharpe", None),
            m.get("Sortino", None),
            m.get("Max Drawdown", None),
            m.get("CAGR", None),
            m.get("Win Rate", None),
            m.get("Total Trades", None),
            m.get("Winning Trades", None),
            m.get("Losing Trades", None),
            m.get("Avg Win %", None),
            m.get("Avg Holding Days", None),
            m.get("Calmar Ratio", None),
            m.get("Net Profit", None)  # Gross Profit as another metric
        ])
    else:
        rows.append([k] + [None] * 14)  # Adjusted for 14 metrics
df_summary = pd.DataFrame(rows, columns=[
    "file", "ann_ret", "ann_vol", "sharpe", "sortino", "mdd",
    "cagr", "win_rate", "total_trades", "winning_trades", "losing_trades",
    "avg_win", "avg_holding", "calmar", "net_profit"
])

print(df_summary)


=== Summary Table ===
        file   ann_ret   ann_vol    sharpe    sortino       mdd      cagr  \
0  TICK1.csv  1.631405  0.197178  4.802183  14.989015 -0.095049  1.628795   
1  TICK2.csv  4.408016  2.389580  1.054925  10.722631 -0.278936  4.397758   
2  TICK3.csv  2.302074  0.199195  5.899988  24.582854 -0.049671  2.297548   
3  TICK4.csv  0.771201  0.294014  1.954279   2.508594 -0.389616  0.769840   

    win_rate  total_trades  winning_trades  losing_trades   avg_win  \
0  58.385093           644             376            268  1.335428   
1  63.790186           591             377            214  2.795902   
2  64.297254           619             398            221  1.295229   
3  48.016701           479             230            249  1.493953   

   avg_holding     calmar  net_profit  
0     1.513975  17.163804  381.989368  
1     1.504230  15.802965  902.889229  
2     1.407108  46.346470  420.040887  
3     1.553236   1.979386  181.625196  
